In [1]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications
build_and_test.sh   LICENSE	    README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	    SECURITY.md  tests
experiments	    pyproject.toml  src		 test.sh


In [2]:
!pip install -e .

Obtaining file:///content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.9 MB/s eta 0:00:00
  Building editable for transformercompression (pyproject.toml) ... done
  Created wheel for transformercompression: filename=transformercompression-0.0.1-0.editable-py3-none-any.

In [2]:
import slicegpt
from slicegpt import rotate, model_utils
print("slicegpt imported:", slicegpt.__file__)
print("rotate imported:", rotate.__file__)
print("model_utils imported:", model_utils.__file__)

slicegpt imported: /content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications/src/slicegpt/__init__.py
rotate imported: /content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications/src/slicegpt/rotate.py
model_utils imported: /content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications/src/slicegpt/model_utils.py


In [3]:
MODEL_HF_CODE = "Qwen/Qwen2-1.5B-Instruct"
MODEL_ID = "Qwen2-1.5B-Instruct"

In [4]:
import os, textwrap

# Where to save logs and models in your Drive
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, f"logs_{MODEL_ID}")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, f"models_{MODEL_ID}")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# sparsities = [0.0, 0.10, 0.25, 0.4, 0.6]
# datasets = ["wikitext2"]

print("Logs in:", LOG_DIR)
print("Models in:", MODEL_DIR)

Logs in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_Qwen2-1.5B-Instruct
Models in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_Qwen2-1.5B-Instruct


In [5]:
# Cell 1: Slicing with correct dtype
import os
import subprocess
from datetime import datetime

def run_slicegpt(dataset, sparsity):
    log_name = f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}.txt".replace(".", "p")
    log_path = os.path.join(LOG_DIR, log_name)

    # if os.path.exists(log_path):
    #     print(f"[SKIP] Log already exists for {dataset}, sparsity={sparsity}: {log_path}")
    #     return

    save_dir = os.path.join(MODEL_DIR, f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}".replace(".", "p"))
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python",
        "/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", MODEL_HF_CODE,
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--no-wandb",
        "--cal-batch-size", "8"
        # Removed --dtype flag - will use model's default (fp16 for Gemma)
    ]

    print("\n=====================================================")
    print("Running:", " ".join(cmd))
    print("Log file:", log_path)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w") as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
            f.write(line)

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())

In [6]:
sparsities = [0, 0.1, 0.25, 0.4, 0.6]
datasets = ["wikitext2", "squad", "hotpotqa", "coqa"]

for dataset in datasets:
    for s in sparsities:
        run_slicegpt(dataset, s)


Running: python /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py --model Qwen/Qwen2-1.5B-Instruct --cal-dataset wikitext2 --save-dir /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_Qwen2-1.5B-Instruct/Qwen2-1p5B-Instruct_wikitext2_s0p00 --sparsity 0 --device cuda:0 --no-wandb --cal-batch-size 8
Log file: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_Qwen2-1.5B-Instruct/Qwen2-1p5B-Instruct_wikitext2_s0p00ptxt
Start: 2026-01-19 00:14:46.687069

Running SliceGPT experiment.
PyTorch device: cuda:0
Number of available cuda devices: 1
Loading Qwen/Qwen2-1.5B-Instruct config and model weights from Hugging Face
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading model done
Loading dataset: wikitext2
Loading dataset done
Preparing dataloader
Preparing dataloader done
Preparing test dataloader
Token indices sequence length is longer than the specified maxi